# Scaling ERDDAP Obs Matching

This notebook is a demonstration of a method to scale the collection SalishSeaCast model field values to match
ocean observations for large numbers of observations.
It builds on the foundation of the `BasicERDDAP-ObsMatching.ipynb` notebook in this directory to execute ERDDAP server requests in parallel.
It uses `dask` to partition the `pandas.Dataframe` containing the observations into sections which can be processed concurrently
and combined to produce the final dataframe that combines the model field values closest to the observations with the observations.
The observations are assumed to be defined by a collection of 4d time-space coordinates in a `pandas.DataFrame`.
(Those could, of course, be read from a CSV file.)

It is assumed that the observations are discrete and independent;
i.e. single points in time-space.

Extraction of time series, depth profiles, or other hyperslabs of model fields would need to be done differently.
For members of the MOAD group who have access to the model results file storage those things are best done using
[Reshapr](https://reshapr.readthedocs.io/en/latest/).

The conda environment file used to run this notebook is `environment.yaml` in this directory.

In [2]:
import dask.dataframe as dd
import numpy as np
import pandas as pd
import xarray as xr
from zmq.utils.buffers import asbuffer_w

For reference, the versions of Python and the packages that are important to what we are doing
that were in the environment in which this notebook was last run are:

In [3]:
import sys
import netCDF4
import numpy
import pandas
import xarray

print(f"Python {sys.version=}")
print(f"{numpy.__version__=}")
print(f"{xarray.__version__=}")
print(f"{pandas.__version__=}")
print(f"{netCDF4.__version__=}")

Python sys.version='3.13.2 | packaged by conda-forge | (main, Feb 17 2025, 14:10:22) [GCC 13.3.0]'
numpy.__version__='2.2.3'
xarray.__version__='2025.1.2'
pandas.__version__='2.2.3'
netCDF4.__version__='1.7.2'


## ERDDAP Dataset Setup

As we did in the `BasicERDDAP-ObsMatching.ipynb` notebook,
we load the metadata for the ERDDAP dataset we are interested in,
and reduce the set of variables we'll request to those of interest.

In [4]:
erddap_url = "https://salishsea.eos.ubc.ca/erddap"
dataset_id = "ubcSSg3DPhysicsFields1hV21-11"
dataset_url = f"{erddap_url}/griddap/{dataset_id}"

ds = xr.open_dataset(dataset_url)

In [5]:
all_vars = set(ds.data_vars)
keep_vars = {"salinity", "temperature"}
drop_vars = all_vars - keep_vars
ds = xr.open_dataset(dataset_url, drop_variables=drop_vars)

## Function to Convert ERDDAP Result to Dataframe

This is a copy of the function that we developed in the `BasicERDDAP-ObsMatching.ipynb` notebook.

In [6]:
def get_model_values_df(ds, time, depth, gridY, gridX):
    ds_pt = (ds
        .sel(time=time, method="nearest")
        .sel(depth=depth, method="nearest")
        .sel(gridY=gridY, gridX=gridX)
    )
    df_pt = pd.DataFrame(
        {
            "time": [ds_pt.time.values],
            "depth": [ds_pt.depth.item()],
        }
    )
    for var in ds_pt.data_vars:
        df_pt[var] = [ds_pt[var].item()]
    return df_pt

## Generate Random "Observations" for Testing

In [9]:
def random_timestamps(start, end, n=10):

    start_u = start.value//10**9
    end_u = end.value//10**9

    return pd.to_datetime(np.random.randint(start_u, end_u, n), unit='s')

In [10]:
start = pd.to_datetime('2015-01-01')
end = pd.to_datetime('2018-01-01')
random_timestamps(start, end)

DatetimeIndex(['2016-05-01 17:20:41', '2015-07-25 18:09:41',
               '2015-10-07 01:00:59', '2015-10-27 15:50:27',
               '2015-04-09 04:29:39', '2015-02-24 03:55:11',
               '2016-02-04 14:35:01', '2015-05-20 17:50:43',
               '2017-02-17 14:48:12', '2017-04-13 20:08:24'],
              dtype='datetime64[ns]', freq=None)

In [66]:
start = pd.to_datetime('2007-01-01')
end = pd.to_datetime('2025-03-10')
n_obs = 10
obs_df = pd.DataFrame(
    {
        "time": random_timestamps(start, end, n=n_obs),
        "depth": np.random.random_sample(n_obs) * 3.5,
        "gridY": np.random.randint(400, 498+1, n_obs),
        "gridX": np.random.randint(254, 300+1, n_obs),
        "salinity": np.random.random_sample(n_obs) * 5 + 25,
        "temperature": np.random.random_sample(n_obs) * 10 + 5,
    }
)

obs_df

,time,depth,gridY,gridX,salinity,temperature
0,2024-07-29 11:52:49,2.071569,480,286,28.108223,10.020058
1,2011-07-10 22:01:26,0.686509,447,258,27.445904,10.323401
2,2014-11-29 09:57:23,2.148633,455,260,26.726839,12.650015
3,2018-07-26 16:43:20,1.209171,468,270,27.031095,14.994867
4,2021-08-02 14:07:36,3.031494,406,277,27.452107,13.522278
5,2015-12-25 06:54:05,1.441042,413,276,27.859430,9.165618
6,2023-01-12 18:49:27,1.243827,469,288,26.639017,6.072582
7,2014-09-16 12:55:54,2.539084,414,291,25.205296,6.686449
8,2024-05-23 07:24:58,0.902473,476,271,26.238904,9.531494
9,2013-07-26 15:08:48,1.167660,449,299,29.736053,12.236038


In [114]:
obs_df = obs_df.sort_values(by="time", ignore_index=True)

obs_df


,time,depth,gridY,gridX,salinity,temperature
0,2007-12-12 15:11:27,2.269713,483,271,26.453434,12.992669
1,2008-04-05 02:51:54,1.447615,474,278,27.855686,14.694386
2,2009-08-31 04:24:32,0.913275,416,293,25.038084,6.449187
3,2010-01-05 21:11:00,0.730401,485,284,26.130166,6.652713
4,2010-09-16 06:14:53,3.408327,406,272,26.515946,12.162864
5,2012-03-15 19:30:21,1.319692,497,299,28.042399,5.256601
6,2014-11-04 14:39:59,3.344971,447,262,25.949274,11.888787
7,2016-09-12 03:21:02,1.710429,402,294,25.736993,6.679538
8,2017-03-30 10:32:58,1.884861,476,256,27.014221,7.200219
9,2017-08-13 06:05:01,2.465039,495,261,29.750517,13.766072


In [117]:
model_dfs = [
    get_model_values_df(ds, time, depth, gridY, gridX)
    for time, depth, gridY, gridX, *_ in obs_df.to_numpy()
]

In [118]:
model_df = pd.concat(model_dfs, ignore_index=True)

In [119]:
result_df = obs_df.join(model_df, rsuffix="_model")

result_df

,time,depth,gridY,gridX,salinity,temperature,time_model,depth_model,salinity_model,temperature_model
0,2007-12-12 15:11:27,2.269713,483,271,26.453434,12.992669,2007-12-12 15:30:00,2.500011,25.218927,5.754432
1,2008-04-05 02:51:54,1.447615,474,278,27.855686,14.694386,2008-04-05 02:30:00,1.500003,25.632439,8.384904
2,2009-08-31 04:24:32,0.913275,416,293,25.038084,6.449187,2009-08-31 04:30:00,0.500000,18.271185,18.620918
3,2010-01-05 21:11:00,0.730401,485,284,26.130166,6.652713,2010-01-05 21:30:00,0.500000,25.607563,6.063612
4,2010-09-16 06:14:53,3.408327,406,272,26.515946,12.162864,2010-09-16 06:30:00,3.500031,27.935551,13.915732
5,2012-03-15 19:30:21,1.319692,497,299,28.042399,5.256601,2012-03-15 19:30:00,1.500003,27.233837,6.877106
6,2014-11-04 14:39:59,3.344971,447,262,25.949274,11.888787,2014-11-04 14:30:00,3.500031,25.747776,11.319856
7,2016-09-12 03:21:02,1.710429,402,294,25.736993,6.679538,2016-09-12 03:30:00,1.500003,25.725342,13.524038
8,2017-03-30 10:32:58,1.884861,476,256,27.014221,7.200219,2017-03-30 10:30:00,1.500003,27.663313,8.043854
9,2017-08-13 06:05:01,2.465039,495,261,29.750517,13.766072,2017-08-13 06:30:00,2.500011,25.464815,20.213211


In [120]:
obs_ddf = dd.from_pandas(obs_df, npartitions=2)

In [7]:
def get_model_values_df_dask(ds, row):
    ds_pt = (ds
        .sel(time=row.time, method="nearest")
        .sel(depth=row.depth, method="nearest")
        .sel(gridY=row.gridY, gridX=row.gridX)
    )
    df_pt = pd.DataFrame(
        {
            "time": [ds_pt.time.values],
            "depth": [ds_pt.depth.item()],
        }
    )
    for var in ds_pt.data_vars:
        df_pt[var] = [ds_pt[var].item()]
    return df_pt

In [122]:
model_dfs = [get_model_values_df_dask(ds, row) for row in obs_ddf.itertuples()]

In [123]:
model_df = pd.concat(model_dfs, ignore_index=True)

In [124]:
result_df = obs_df.join(model_df, rsuffix="_model")

result_df

,time,depth,gridY,gridX,salinity,temperature,time_model,depth_model,salinity_model,temperature_model
0,2007-12-12 15:11:27,2.269713,483,271,26.453434,12.992669,2007-12-12 15:30:00,2.500011,25.218927,5.754432
1,2008-04-05 02:51:54,1.447615,474,278,27.855686,14.694386,2008-04-05 02:30:00,1.500003,25.632439,8.384904
2,2009-08-31 04:24:32,0.913275,416,293,25.038084,6.449187,2009-08-31 04:30:00,0.500000,18.271185,18.620918
3,2010-01-05 21:11:00,0.730401,485,284,26.130166,6.652713,2010-01-05 21:30:00,0.500000,25.607563,6.063612
4,2010-09-16 06:14:53,3.408327,406,272,26.515946,12.162864,2010-09-16 06:30:00,3.500031,27.935551,13.915732
5,2012-03-15 19:30:21,1.319692,497,299,28.042399,5.256601,2012-03-15 19:30:00,1.500003,27.233837,6.877106
6,2014-11-04 14:39:59,3.344971,447,262,25.949274,11.888787,2014-11-04 14:30:00,3.500031,25.747776,11.319856
7,2016-09-12 03:21:02,1.710429,402,294,25.736993,6.679538,2016-09-12 03:30:00,1.500003,25.725342,13.524038
8,2017-03-30 10:32:58,1.884861,476,256,27.014221,7.200219,2017-03-30 10:30:00,1.500003,27.663313,8.043854
9,2017-08-13 06:05:01,2.465039,495,261,29.750517,13.766072,2017-08-13 06:30:00,2.500011,25.464815,20.213211


In [46]:
start = pd.to_datetime('2007-01-01')
end = pd.to_datetime('2025-03-10')
n_obs, n_parts = 90, 3
obs_df = pd.DataFrame(
    {
        "time": random_timestamps(start, end, n=n_obs),
        "depth": np.random.random_sample(n_obs) * 3.5,
        "gridY": np.random.randint(400, 498+1, n_obs),
        "gridX": np.random.randint(254, 300+1, n_obs),
        "salinity": np.random.random_sample(n_obs) * 5 + 25,
        "temperature": np.random.random_sample(n_obs) * 10 + 5,
    }
)
obs_df = obs_df.sort_values(by="time", ignore_index=True)

In [47]:
obs_ddf = dd.from_pandas(obs_df, npartitions=n_parts)

In [142]:
model_dfs = [get_model_values_df_dask(ds, row) for row in obs_ddf.itertuples()]

In [143]:
model_df = pd.concat(model_dfs, ignore_index=True)

In [144]:
result_df = obs_df.join(model_df, rsuffix="_model")

result_df

,time,depth,gridY,gridX,salinity,temperature,time_model,depth_model,salinity_model,temperature_model
0,2007-05-23 05:49:43,1.714873,411,292,29.122525,14.070561,2007-05-23 05:30:00,1.500003,25.034149,11.312622
1,2008-10-20 23:47:43,0.236672,462,293,26.450010,11.950840,2008-10-20 23:30:00,0.500000,26.592619,11.199994
2,2010-02-26 21:06:24,0.487208,494,271,28.330895,6.784119,2010-02-26 21:30:00,0.500000,28.018591,7.878107
3,2011-02-19 00:40:58,2.929656,409,269,28.198396,14.633318,2011-02-19 00:30:00,2.500011,28.868555,7.414166
4,2011-04-23 10:35:11,0.502289,449,266,29.365020,11.502277,2011-04-23 10:30:00,0.500000,27.365599,9.711308
5,2011-07-22 08:59:20,1.266480,416,260,29.564699,9.591357,2011-07-22 08:30:00,1.500003,5.911692,18.302515
6,2012-04-03 10:17:36,0.370950,497,279,28.585724,6.869343,2012-04-03 10:30:00,0.500000,26.525877,7.551504
7,2013-04-22 04:02:19,2.066436,403,297,25.104015,8.915872,2013-04-22 04:30:00,2.500011,29.239824,8.774421
8,2013-07-10 09:14:36,0.705584,439,259,29.366658,5.978880,2013-07-10 09:30:00,0.500000,21.114786,17.749380
9,2013-09-20 10:56:21,2.737289,437,269,29.711142,13.731994,2013-09-20 10:30:00,2.500011,19.791355,16.567026


In [48]:
from dask.distributed import LocalCluster, Client
cluster = LocalCluster(n_workers=3, threads_per_worker=8, processes=True)
client = Client(cluster)

In [49]:
model_dfs = [get_model_values_df_dask(ds, row) for row in obs_ddf.itertuples()]
model_df = pd.concat(model_dfs, ignore_index=True)
result_df = obs_df.join(model_df, rsuffix="_model")

result_df

,time,depth,gridY,gridX,salinity,temperature,time_model,depth_model,salinity_model,temperature_model
0,2007-05-23 10:15:18,1.696266,458,293,28.875133,5.194214,2007-05-23 10:30:00,1.500003,11.532490,14.946426
1,2007-09-20 01:33:25,2.898238,487,269,27.291126,12.189823,2007-09-20 01:30:00,2.500011,25.601780,15.195985
2,2007-12-24 07:42:13,2.467887,434,291,27.223816,5.944019,2007-12-24 07:30:00,2.500011,22.180975,6.700281
3,2008-02-25 22:09:37,0.392468,402,255,26.295819,11.815650,2008-02-25 22:30:00,0.500000,25.549454,6.858864
4,2008-03-23 05:51:34,0.551444,486,260,28.548815,7.065965,2008-03-23 05:30:00,0.500000,28.130602,7.733637
...,...,...,...,...,...,...,...,...,...,...
85,2024-01-06 01:37:21,2.346718,441,300,25.958553,5.014811,2024-01-06 01:30:00,2.500011,22.636038,7.267166
86,2024-02-06 14:03:14,0.510461,485,272,26.287431,11.568225,2024-02-06 14:30:00,0.500000,13.368867,4.821707
87,2024-05-08 02:36:16,2.936532,403,261,27.537432,9.657496,2024-05-08 02:30:00,2.500011,23.803493,11.499825
88,2024-05-18 17:04:04,0.906006,418,283,29.544106,11.671529,2024-05-18 17:30:00,0.500000,16.659845,11.802532


In [50]:
client.close()
cluster.close()

In [40]:
model_dfs = [
    get_model_values_df(ds, time, depth, gridY, gridX)
    for time, depth, gridY, gridX, *_ in obs_df.to_numpy()
]
model_df = pd.concat(model_dfs, ignore_index=True)
result_df = obs_df.join(model_df, rsuffix="_model")

result_df

,time,depth,gridY,gridX,salinity,temperature,time_model,depth_model,salinity_model,temperature_model
0,2007-02-01 23:09:40,2.039685,431,261,29.019204,6.903151,2007-02-01 23:30:00,2.500011,27.299467,6.475811
1,2007-02-07 08:12:24,1.593547,424,264,27.437956,8.410979,2007-02-07 08:30:00,1.500003,26.324528,6.602376
2,2007-02-16 21:37:18,0.497895,435,277,27.536166,11.133932,2007-02-16 21:30:00,0.500000,21.994833,6.705160
3,2007-02-21 21:04:15,1.974326,422,259,26.086163,8.994523,2007-02-21 21:30:00,1.500003,25.754135,7.162727
4,2007-08-20 13:35:21,2.626669,494,274,25.425995,10.092070,2007-08-20 13:30:00,2.500011,21.464130,19.398712
...,...,...,...,...,...,...,...,...,...,...
85,2024-03-30 07:46:05,1.429505,402,267,29.721546,10.786255,2024-03-30 07:30:00,1.500003,28.639294,8.761847
86,2024-04-13 16:57:50,0.745775,420,296,25.205243,6.425426,2024-04-13 16:30:00,0.500000,17.551989,8.742970
87,2024-06-20 18:50:49,2.288572,471,282,27.498523,8.483775,2024-06-20 18:30:00,2.500011,11.897174,17.041817
88,2024-10-04 23:51:00,1.926701,428,281,29.628412,13.728851,2024-10-04 23:30:00,1.500003,25.624271,13.701365


In [1]:
import asyncio

async def count():
    print("One")
    await asyncio.sleep(1)
    print("Two")

async def main():
    await asyncio.gather(count(), count(), count())

await main()

One
One
One
Two
Two
Two


In [7]:
async def get_model_values_df(ds, time, depth, gridY, gridX):
    ds_pt = (ds
        .sel(time=time, method="nearest")
        .sel(depth=depth, method="nearest")
        .sel(gridY=gridY, gridX=gridX)
    )
    df_pt = pd.DataFrame(
        {
            "time": [ds_pt.time.values],
            "depth": [ds_pt.depth.item()],
        }
    )
    for var in ds_pt.data_vars:
        df_pt[var] = [ds_pt[var].item()]
    return df_pt

In [8]:
await count()

One
Two


In [15]:
obs_df.head(1).to_numpy()

array([[Timestamp('2010-05-19 15:56:38'), 2.552866200557356, 479, 257,
        27.968327280709598, 10.306808376616068]], dtype=object)

In [16]:
time, depth, gridY, gridX, *_ = obs_df.head(1).to_numpy()[0]

In [18]:
await get_model_values_df(ds, time, depth, gridY, gridX)

,time,depth,salinity,temperature
0,2010-05-19 15:30:00,2.500011,26.75877,13.110973


In [23]:
async def main():
    model_dfs = [
        await get_model_values_df(ds, time, depth, gridY, gridX)
        for time, depth, gridY, gridX, *_ in obs_df.to_numpy()
    ]
    return model_dfs

In [24]:
await main()

[                 time     depth  salinity  temperature
 0 2010-05-19 15:30:00  2.500011  26.75877    13.110973,
                  time     depth   salinity  temperature
 0 2019-01-22 20:30:00  3.500031  27.547913     7.119268,
                  time     depth   salinity  temperature
 0 2009-08-10 03:30:00  2.500011  27.063076    17.572262,
                  time     depth   salinity  temperature
 0 2021-02-16 01:30:00  1.500003  25.445696     4.732948,
                  time  depth   salinity  temperature
 0 2018-10-20 18:30:00    0.5  16.328741    11.954584,
                  time  depth   salinity  temperature
 0 2015-09-24 13:30:00    0.5  16.630058    13.882245,
                  time     depth   salinity  temperature
 0 2019-07-25 10:30:00  1.500003  18.596416    16.294794,
                  time     depth   salinity  temperature
 0 2020-07-19 05:30:00  2.500011  13.045879    18.344345,
                  time  depth  salinity  temperature
 0 2023-05-19 13:30:00    0.5  4.206067  

In [29]:
model_dfs = [
    await get_model_values_df(ds, time, depth, gridY, gridX)
    for time, depth, gridY, gridX, *_ in obs_df.to_numpy()
]

In [30]:
obs_df.head(2).tail(1).to_numpy()

array([[Timestamp('2023-01-26 12:12:49'), 3.1360637158345206, 462, 292,
        27.788554868950158, 14.610275522159284]], dtype=object)

In [37]:
async def foo(n):
    time, depth, gridY, gridX, *_ = obs_df.head(n).to_numpy()[0]
    return await get_model_values_df(ds, time, depth, gridY, gridX)

In [38]:
await foo(1)

,time,depth,salinity,temperature
0,2013-12-04 05:30:00,2.500011,27.384897,8.12785


In [39]:
await foo(2)

,time,depth,salinity,temperature
0,2013-12-04 05:30:00,2.500011,27.384897,8.12785


In [47]:
async def main():
    await asyncio.gather(foo(1), foo(2), foo(3))

In [48]:
await main()

In [52]:
await asyncio.gather(foo(1), foo(2), foo(3))

[                 time  depth   salinity  temperature
 0 2015-09-24 17:30:00    0.5  15.332047    14.039825,
                  time  depth   salinity  temperature
 0 2015-09-24 17:30:00    0.5  15.332047    14.039825,
                  time  depth   salinity  temperature
 0 2015-09-24 17:30:00    0.5  15.332047    14.039825]

In [67]:
await asyncio.gather(*(foo(i) for i in range(1, 11)))

[                 time     depth   salinity  temperature
 0 2024-07-29 11:30:00  2.500011  20.948299    20.099245,
                  time     depth   salinity  temperature
 0 2024-07-29 11:30:00  2.500011  20.948299    20.099245,
                  time     depth   salinity  temperature
 0 2024-07-29 11:30:00  2.500011  20.948299    20.099245,
                  time     depth   salinity  temperature
 0 2024-07-29 11:30:00  2.500011  20.948299    20.099245,
                  time     depth   salinity  temperature
 0 2024-07-29 11:30:00  2.500011  20.948299    20.099245,
                  time     depth   salinity  temperature
 0 2024-07-29 11:30:00  2.500011  20.948299    20.099245,
                  time     depth   salinity  temperature
 0 2024-07-29 11:30:00  2.500011  20.948299    20.099245,
                  time     depth   salinity  temperature
 0 2024-07-29 11:30:00  2.500011  20.948299    20.099245,
                  time     depth   salinity  temperature
 0 2024-07-29 11:30:00 